# Hedge-Frequency and Transaction-Cost Scenario Analysis

This notebook compares hedge frequencies and transaction-cost assumptions. The goal is to show the tradeoff between risk reduction and trading cost.

## Concepts

Scenario analysis means changing assumptions and comparing results. Here, the assumptions are hedge frequency and transaction cost.

Sensitivity analysis asks how much the output changes when one input changes. Here, the outputs are hedging error, tail outcomes, and transaction costs.

A hedge-frequency comparison asks whether hedging more often improves the hedging error distribution. More frequent hedging can reduce risk from Delta drift, but it can also increase transaction costs.

A cost-risk frontier compares risk against cost. A better scenario has lower risk for the same cost, or lower cost for the same risk.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.hedging import HedgeContract
from src.scenario_analysis import (
    build_scenario_figures,
    run_scenario_grid,
    save_scenario_outputs,
    summarize_scenarios,
)
from src.simulation import simulate_gbm_paths

## Simulation and contract inputs

In [ ]:
starting_price = 100.0
strike = 100.0
risk_free_rate = 0.04
volatility = 0.25
maturity_years = 30 / 365.25

steps = 30
num_paths = 3_000
random_seed = 42

transaction_cost_bps_values = [0, 1, 5, 10]

In [ ]:
paths = simulate_gbm_paths(
    starting_price=starting_price,
    drift=risk_free_rate,
    volatility=volatility,
    time_horizon=maturity_years,
    steps=steps,
    num_paths=num_paths,
    random_seed=random_seed,
)

contract = HedgeContract(
    option_type="call",
    strike=strike,
    maturity_years=maturity_years,
    risk_free_rate=risk_free_rate,
    volatility=volatility,
    position=1,
)

paths.head()

## Run scenario grid

In [ ]:
scenario_results = run_scenario_grid(
    paths=paths,
    contract=contract,
    transaction_cost_bps_values=transaction_cost_bps_values,
    include_event_trigger=True,
    event_delta_threshold=0.10,
)

scenario_results.head()

## Scenario summary table

In [ ]:
scenario_summary = summarize_scenarios(scenario_results)
scenario_summary

## Save tables and figures

In [ ]:
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"

saved_tables = save_scenario_outputs(
    scenario_results=scenario_results,
    scenario_summary=scenario_summary,
    output_dir=processed_dir,
)

saved_figures = build_scenario_figures(
    scenario_summary=scenario_summary,
    output_dir=figures_dir,
)

saved_tables, saved_figures

## Interpretation

The table and charts show why more frequent hedging is not automatically better. Daily hedging may lower the standard deviation of hedging error, but it also trades more often. When transaction costs rise, the cost of frequent rebalancing can offset some of the risk reduction.

The worst 5 percent outcome is represented by the 5th percentile hedging error. This helps describe tail risk instead of only looking at the mean.